In [1]:
from fetchers.fetcher import Fetcher, FMPFetcher
from cleaners.cleaner import Cleaner, FMPCleaner
from config.config import ROOT
from config.fmp_endpoint import fmp_update_endpoints
from config.schema_config import schema_map
import pandas as pd 
import os 
from dotenv import load_dotenv
import glob

In [2]:
load_dotenv()
api = os.getenv('FMP_API')

In [3]:
symbols = pd.read_csv(f"{ROOT}/data/cleaned/profile.csv").sort_values('market_cap', ascending=False)
symbols = symbols['ticker'].tolist()
symbols = symbols[:10]

In [4]:
def fmp_pipeline(symbols, api, endpoints, file_path, schemamap=schema_map, fetch_all=True):
    fmp = FMPFetcher(symbols,api)
    fmp_clean = FMPCleaner(root=file_path)
    clean = Cleaner(root=file_path)
    files = glob.glob(f"{file_path}/cleaned/*.csv")
    if fetch_all:
        fmp.fetch_all(file_path=file_path, endpoint_config=endpoints)
    else:
        fmp.fetch_endpoints(enpoint_lst=endpoints, file_path=file_path)
    fmp_clean.keep_and_rename(schema_map=schemamap, input_file = file_path)
    for file in files:
        df = pd.read_csv(file)
        clean.data_cleaning(df, existent=False)
        

In [5]:
fmp_pipeline(symbols, api, endpoints=fmp_update_endpoints, file_path=f"{ROOT}/notebooks/data")